In [1]:
import sympy as sp

# Initialize printing for clear math rendering in Jupyter/terminals
sp.init_printing(use_unicode=True)

# 1. Pathway & Physical Infrastructure Fill Ratios

## 1.1 Conduit Fill Ratio (NEC Chapter 9, Table 1 & BICSI TDMM)

Conduit fill limits prevent excessive pulling tension, jacket damage, and heat accumulation during cable installations. The National Electrical Code (NEC) specifies maximum cross-sectional area limits based on conductor count:

* 1 Cable: $53\%$ maximum fill
* 2 Cables: $31\%$ maximum fill (derated to prevent cable jamming when cables twist)
* 3 or more Cables: $40\%$ maximum fill

$$\text{Fill}_{\%} = \left( \frac{\sum_{i=1}^{n} \frac{\pi \cdot d_i^2}{4}}{\frac{\pi \cdot D_{\text{inner}}^2}{4}} \right) \times 100 = \left( \frac{A_{\text{cables}}}{A_{\text{conduit}}} \right) \times 100$$For $N$ identical cables of outer diameter $d$:$$\text{Fill}_{\%} = \left( \frac{N \cdot d^2}{D_{\text{inner}}^2} \right) \times 100$$

### Code

In [4]:
# Conduit Fill Symbols
d_cable, D_inner, A_cables_total, Fill_pct = sp.symbols(
    'd_cable D_inner A_cables_total Fill_pct', positive=True, real=True
)
N_cables = sp.symbols('N_cables', integer=True, positive=True)

# Equations
eq_conduit_area = sp.Eq(sp.Symbol('A_conduit'), sp.pi * (D_inner / 2)**2)
eq_identical_conduit_fill = sp.Eq(
    Fill_pct, 
    (N_cables * (sp.pi * (d_cable / 2)**2) / (sp.pi * (D_inner / 2)**2)) * 100
)
eq_general_conduit_fill = sp.Eq(
    Fill_pct, 
    (A_cables_total / (sp.pi * (D_inner / 2)**2)) * 100
)

# Analytical Solutions
# 1. Minimum conduit inner diameter required for a specified cable bundle area and fill target:
min_D_inner = sp.solve(eq_general_conduit_fill, D_inner)[0]

# 2. Maximum number of uniform cables allowable in a given conduit:
max_N_conduit_cables = sp.solve(eq_identical_conduit_fill, N_cables)[0]

eq_identical_conduit_fill

                               2
           100⋅N_cables⋅d_cable 
Fill_pct = ─────────────────────
                        2       
                  Dᵢₙₙₑᵣ        

### Examples

## 1.2 Cable Tray Fill (NEC 392 & TIA-569)

Cable tray cross-sectional fill evaluates volume constraints for ladder, ventilated trough, or wire mesh pathways. Design standards (such as ANSI/TIA-569) typically mandate a maximum initial planning fill of $40\%$ to accommodate pathway accessibility and future churn, with an absolute physical limit of $50\%$.$$\text{Tray Fill}_{\%} = \left( \frac{\sum_{k} N_k \cdot \left( \frac{\pi \cdot d_k^2}{4} \right)}{W_{\text{tray}} \cdot H_{\text{tray}}} \right) \times 100$$

### Code

In [6]:
# Cable Tray Fill Symbols
W_tray, H_tray, A_tray, Tray_Fill_pct = sp.symbols(
    'W_tray H_tray A_tray Tray_Fill_pct', positive=True, real=True
)

# Equations
eq_tray_area = sp.Eq(A_tray, W_tray * H_tray)
eq_tray_fill = sp.Eq(Tray_Fill_pct, (A_cables_total / (W_tray * H_tray)) * 100)

# Analytical Solution: Minimum tray width required given cable bundle area, depth, and max fill:
min_W_tray = sp.solve(eq_tray_fill, W_tray)[0]

eq_tray_fill

                100⋅A_cables_total
Tray_Fill_pct = ──────────────────
                  H_tray⋅W_tray   

### Examples

## 1.3 Cable Tray Linear Weight Loading
Physical pathway design requires verifying that total cable weight plus pathway self-weight does not exceed structural support and bracket ratings (such as NEMA Class 12C, 20B, or 20C ratings in pounds per linear foot).$$W_{\text{linear}} = w_{\text{tray}} + \sum_{k} N_k \cdot w_k$$

### Code

In [9]:
# Cable Tray Weight Loading Symbols
w_tray, w_cables_total, W_linear = sp.symbols(
    'w_tray w_cables_total W_linear', positive=True, real=True
)

eq_tray_weight = sp.Eq(W_linear, w_tray + w_cables_total)

# Analytical Solution: Allowable total cable weight capacity given tray structural limit:
allowable_w_cables = sp.solve(eq_tray_weight, w_cables_total)[0]

eq_tray_weight

Wₗᵢₙₑₐᵣ = w_cables_total + w_tray

# 2. Optical Fiber Transmission & Link Engineering

## 2.1 Optical Link Power Budget (TIA-568 / ITU-T G.652)

End-to-end passive insertion loss accounts for glass attenuation across link length $L$, mated connector pairs, fusion or mechanical splices, and engineering design headroom (safety margin).$$\text{Loss}_{\text{total}} = (\alpha_{\lambda} \cdot L) + (N_{\text{conn}} \cdot L_{\text{conn}}) + (N_{\text{splice}} \cdot L_{\text{splice}}) + M_{\text{safety}}$$

### Code

In [19]:
# Optical Loss Budget Symbols

L_fiber, alpha_lambda = sp.symbols('L_fiber alpha_lambda', positive=True, real=True)

N_conn, N_splice = sp.symbols('N_conn N_splice', integer=True, nonnegative=True)

L_conn, L_splice, M_safety, Loss_total = sp.symbols(
    'L_conn L_splice M_safety Loss_total', positive=True, real=True
)

eq_optical_loss = sp.Eq(
    Loss_total, 
    (alpha_lambda * L_fiber) + (N_conn * L_conn) + (N_splice * L_splice) + M_safety
)

# Analytical Solution: Maximum optical reach (L_fiber) given allowable link budget:
max_L_fiber = sp.solve(eq_optical_loss, L_fiber)[0]

eq_optical_loss

Lossₜₒₜₐₗ = L_conn⋅N_conn + L_fiber⋅α_λ + L_splice⋅N_splice + M_safety